In [1]:
# Importamos las librerías necesarias: os para variables de entorno, pandas para mostrar resultados y el conector de Snowflake.

import os
import pandas as pd
import snowflake.connector

Definimos constantes importantes: Tablas de Stage (origen) y OBT_TRIPS (destino), además de los esquemas en Snowflake.

In [2]:
DATABASE = os.getenv("SNOWFLAKE_DATABASE")
ANALYTICS_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS")

STAGE_TABLE = "TRIPS_ENRICHED_UNIFIED_STAGE"
OBT_TABLE = "OBT_TRIPS"

RESULTS_03 = []

Establecemos la conexión conectándonos a la base de datos y verificando que el esquema de Analytics esté disponible.

In [3]:
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=DATABASE,
    role=os.getenv("SNOWFLAKE_ROLE"),
)

cur = conn.cursor()
cur.execute(f"USE DATABASE {DATABASE}")
cur.execute(f"CREATE SCHEMA IF NOT EXISTS {ANALYTICS_SCHEMA}")

print("Snowflake connection ready")

Snowflake connection ready


Creamos la estructura de la 'One Big Table' (OBT_TRIPS). Esta será la tabla analítica final y contendrá métricas calculadas y tipos correctos.

In [4]:
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {ANALYTICS_SCHEMA}.{OBT_TABLE} (
    --Tiempo
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    pickup_date DATE,
    pickup_hour INTEGER,
    dropoff_date DATE,
    dropoff_hour INTEGER,
    day_of_week INTEGER,
    month INTEGER,
    year INTEGER,

    --Ubicacion
    pu_location_id INTEGER,
    pu_zone STRING,
    pu_borough STRING,
    do_location_id INTEGER,
    do_zone STRING,
    do_borough STRING,
    
    --Servicio y codigos
    service_type STRING,
    vendor_id INTEGER,
    vendor_name STRING,
    rate_code_id INTEGER,
    rate_code_desc STRING,
    payment_type INTEGER,
    payment_type_desc STRING,
    trip_type INTEGER,

    --Viajes
    passenger_count FLOAT,
    trip_distance FLOAT,
    store_and_fwd_flag STRING,

    --Tarifas
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    improvement_surcharge FLOAT,
    congestion_surcharge FLOAT,
    airport_fee FLOAT,
    ehail_fee FLOAT,
    total_amount FLOAT,

    -- Derivadas
    trip_duration_min FLOAT,
    avg_speed_mph FLOAT,
    tip_pct FLOAT,

    --Calidad
    run_id STRING,
    ingested_at_utc TIMESTAMP,
    source_year INTEGER,
    source_month INTEGER
)
""")

print("OBT table ready")

OBT table ready


Creamos una función auxiliar que elimina los viajes para un mes/año y tipo de servicio específico. Esto hace que el proceso sea IDEMPOTENTE.

In [5]:
def delete_obt_month(service_type, year_value, month_value):
    cur.execute(f"""
        DELETE FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE service_type = '{service_type}'
          AND source_year = {year_value}
          AND source_month = {month_value}
    """)

### Función principal 'build_obt_month':
1) Llama a delete_obt_month.
2) Inserta los datos del mes calculado.
LIMPIEZA CLAVES 5 

In [6]:
def build_obt_month(service_type, year_value, month_value):
    delete_obt_month(service_type, year_value, month_value)

    sql = f"""
    INSERT INTO {ANALYTICS_SCHEMA}.{OBT_TABLE}
    SELECT
        pickup_datetime,
        dropoff_datetime,
        CAST(pickup_datetime AS DATE) AS pickup_date,
        EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
        CAST(dropoff_datetime AS DATE) AS dropoff_date,
        EXTRACT(HOUR FROM dropoff_datetime) AS dropoff_hour,
        DAYOFWEEK(pickup_datetime) AS day_of_week,
        EXTRACT(MONTH FROM pickup_datetime) AS month,
        EXTRACT(YEAR FROM pickup_datetime) AS year,

        pu_location_id,
        COALESCE(pu_zone, 'Unknown') AS pu_zone,
        COALESCE(pu_borough, 'Unknown') AS pu_borough,
        do_location_id,
        COALESCE(do_zone, 'Unknown') AS do_zone,
        COALESCE(do_borough, 'Unknown') AS do_borough,

        service_type,
        vendor_id,
        vendor_name,
        COALESCE(rate_code_id, 99) AS rate_code_id,
        rate_code_desc,
        COALESCE(payment_type, 5) AS payment_type,
        payment_type_desc,
        trip_type,

        COALESCE(passenger_count, 0) AS passenger_count,
        trip_distance,
        store_and_fwd_flag,

        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        congestion_surcharge,
        airport_fee,
        ehail_fee,
        total_amount,

        ROUND(DATEDIFF('second', pickup_datetime, dropoff_datetime) / 60.0, 2) AS trip_duration_min,

        CASE
            WHEN DATEDIFF('second', pickup_datetime, dropoff_datetime) > 0
                 AND trip_distance > 0
            THEN ROUND(
                trip_distance / (DATEDIFF('second', pickup_datetime, dropoff_datetime) / 3600.0),
                2
            )
            ELSE NULL
        END AS avg_speed_mph,

        CASE
            WHEN fare_amount > 0
            THEN ROUND((tip_amount / fare_amount) * 100.0, 2)
            ELSE NULL
        END AS tip_pct,

        run_id,
        ingested_at_utc,
        source_year,
        source_month

    FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
    WHERE service_type = '{service_type}'
      -- Validacion 0
      AND source_year = {year_value}
      AND source_month = {month_value}
      
      -- Validacion 1: Fechas existen (no nulas)
      AND pickup_datetime IS NOT NULL
      AND dropoff_datetime IS NOT NULL
      
      -- Validacion 2: La fecha de inicio coincide con el archivo de extraccion
      AND EXTRACT(YEAR FROM pickup_datetime) = {year_value}
      AND EXTRACT(MONTH FROM pickup_datetime) = {month_value}
      
      -- Validacion 3: Distancia del viaje debe ser estrictamente positiva
      AND trip_distance > 0
      
      -- Validacion 4: Que no nos cobren negativo en la tarifa (base o total)
      AND fare_amount >= 0
      AND total_amount >= 0
      
      -- Validacion 5: Duracion de viaje positiva o cero
      AND DATEDIFF('second', pickup_datetime, dropoff_datetime) >= 0
      
      -- Validacion 6: Control atípicos para pasajeros (0-9 es lo razonable en un taxi/van)
      AND COALESCE(passenger_count, 0) BETWEEN 0 AND 9

      -- Validacion 7: Exclusión de velocidades hiper-rápidas/absurdas (> 150 mph) debido a saltos de GPS o fallos de reloj
      AND (
          DATEDIFF('second', pickup_datetime, dropoff_datetime) = 0 
          OR (trip_distance / (DATEDIFF('second', pickup_datetime, dropoff_datetime) / 3600.0)) <= 150
      )    """
    
    cur.execute(sql)

Bloque de pruebas manual: Cargamos de manera individual un mes en la OBT_TRIPS (Enero 2024 para green y yellow) para confirmar funcionalidad.

In [7]:
build_obt_month("yellow", 2024, 1)
build_obt_month("green", 2024, 1)

print("Test month loaded into OBT")

Test month loaded into OBT


Consultas de validación: Verificamos los conteos y mostramos una muestra de 10 registros procesados en Enero de 2024.

In [8]:
test_queries = {
    "yellow_2024_01_count": f"""
        SELECT COUNT(*)
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE service_type = 'yellow'
          AND source_year = 2024
          AND source_month = 1
    """,
    "green_2024_01_count": f"""
        SELECT COUNT(*)
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE service_type = 'green'
          AND source_year = 2024
          AND source_month = 1
    """,
    "sample_rows": f"""
        SELECT
            service_type,
            pickup_datetime,
            dropoff_datetime,
            trip_distance,
            total_amount,
            trip_duration_min,
            avg_speed_mph,
            tip_pct
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        WHERE source_year = 2024
          AND source_month = 1
        LIMIT 10
    """
}

for name, query in test_queries.items():
    print(f"\n--- {name} ---")
    cur.execute(query)
    for row in cur.fetchmany(10):
        print(row)


--- yellow_2024_01_count ---
(2869133,)

--- green_2024_01_count ---
(53409,)

--- sample_rows ---
('yellow', datetime.datetime(2024, 1, 27, 0, 16, 42), datetime.datetime(2024, 1, 27, 0, 19, 31), 0.62, 10.1, 2.82, 13.21, 0.0)
('yellow', datetime.datetime(2024, 1, 27, 0, 22, 56), datetime.datetime(2024, 1, 27, 0, 40, 24), 3.7, 27.28, 17.47, 12.71, 12.53)
('yellow', datetime.datetime(2024, 1, 27, 0, 51, 56), datetime.datetime(2024, 1, 27, 1, 0, 20), 1.41, 18.0, 8.4, 10.07, 30.0)
('yellow', datetime.datetime(2024, 1, 27, 0, 34, 10), datetime.datetime(2024, 1, 27, 0, 42, 33), 1.6, 17.25, 8.38, 11.45, 22.5)
('yellow', datetime.datetime(2024, 1, 27, 0, 49, 6), datetime.datetime(2024, 1, 27, 0, 58, 2), 1.59, 16.7, 8.93, 10.68, 9.35)
('yellow', datetime.datetime(2024, 1, 27, 0, 51, 44), datetime.datetime(2024, 1, 27, 1, 0, 8), 1.5, 18.0, 8.4, 10.71, 30.0)
('yellow', datetime.datetime(2024, 1, 27, 0, 49, 20), datetime.datetime(2024, 1, 27, 0, 53, 58), 0.63, 12.5, 4.63, 8.16, 15.38)
('yellow', 

 Bucle principal: Recorre todos los años y meses (2015-2025) y tipos de servicio (yellow, green) procesándolos y guardando su estado de éxito/error en 'RESULTS_03'.

In [9]:
for service_type in ["yellow", "green"]:
    for year_value in range(2015, 2026):
        for month_value in range(1, 13):
            try:
                print(f"Building OBT for {service_type} {year_value}-{month_value:02d}")
                build_obt_month(service_type, year_value, month_value)

                RESULTS_03.append({
                    "service_type": service_type,
                    "year": year_value,
                    "month": month_value,
                    "status": "success",
                    "message": "ok"
                })

                print(f"Done OBT for {service_type} {year_value}-{month_value:02d}")

            except Exception as e:
                RESULTS_03.append({
                    "service_type": service_type,
                    "year": year_value,
                    "month": month_value,
                    "status": "failed",
                    "message": str(e)
                })

                print(f"Failed OBT for {service_type} {year_value}-{month_value:02d}: {e}")

Building OBT for yellow 2015-01
Done OBT for yellow 2015-01
Building OBT for yellow 2015-02
Done OBT for yellow 2015-02
Building OBT for yellow 2015-03
Done OBT for yellow 2015-03
Building OBT for yellow 2015-04
Done OBT for yellow 2015-04
Building OBT for yellow 2015-05
Done OBT for yellow 2015-05
Building OBT for yellow 2015-06
Done OBT for yellow 2015-06
Building OBT for yellow 2015-07
Done OBT for yellow 2015-07
Building OBT for yellow 2015-08
Done OBT for yellow 2015-08
Building OBT for yellow 2015-09
Done OBT for yellow 2015-09
Building OBT for yellow 2015-10
Done OBT for yellow 2015-10
Building OBT for yellow 2015-11
Done OBT for yellow 2015-11
Building OBT for yellow 2015-12
Done OBT for yellow 2015-12
Building OBT for yellow 2016-01
Done OBT for yellow 2016-01
Building OBT for yellow 2016-02
Done OBT for yellow 2016-02
Building OBT for yellow 2016-03
Done OBT for yellow 2016-03
Building OBT for yellow 2016-04
Done OBT for yellow 2016-04
Building OBT for yellow 2016-05
Done OBT

Convertimos la lista de resultados de la ejecución a un DataFrame de Pandas para visualizar fácilmente los meses procesados.

In [10]:
results_03_df = pd.DataFrame(RESULTS_03)
results_03_df

,service_type,year,month,status,message
0,yellow,2015,1,success,ok
1,yellow,2015,2,success,ok
2,yellow,2015,3,success,ok
3,yellow,2015,4,success,ok
4,yellow,2015,5,success,ok
...,...,...,...,...,...
259,green,2025,8,success,ok
260,green,2025,9,success,ok
261,green,2025,10,success,ok
262,green,2025,11,success,ok


Mostramos el resumen de ejecuciones exitosas vs fallidas agrupadas por tipo de servicio.

In [11]:
results_03_df.groupby(["service_type", "status"]).size()

service_type  status 
green         success    132
yellow        success    132
dtype: int64

Exportamos el registro (log) de estado de las cargas a un archivo CSV local 'obt_build_results.csv' y mostramos confirmación.

In [12]:
results_03_df.to_csv("obt_build_results.csv", index=False)
print("Saved results to obt_build_results.csv")

Saved results to obt_build_results.csv


Última validación post-carga: Observamos conteos históricos de años y meses masivos directamente sobre la tabla OBT analítica ya completamente alimentada.

In [13]:
validation_queries = {
    "total_rows": f"""
        SELECT COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
    """,
    "rows_by_service": f"""
        SELECT service_type, COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        GROUP BY service_type
        ORDER BY service_type
    """,
    "rows_by_year": f"""
        SELECT service_type, source_year, COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        GROUP BY service_type, source_year
        ORDER BY service_type, source_year
    """,
    "sample_metrics": f"""
        SELECT
            service_type,
            trip_distance,
            total_amount,
            trip_duration_min,
            avg_speed_mph,
            tip_pct
        FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
        LIMIT 20
    """
}

for name, query in validation_queries.items():
    print(f"\n--- {name} ---")
    cur.execute(query)
    for row in cur.fetchmany(20):
        print(row)


--- total_rows ---
(855253972,)

--- rows_by_service ---
('green', 66867019)
('yellow', 788386953)

--- rows_by_year ---
('green', 2015, 18904338)
('green', 2016, 16113247)
('green', 2017, 11561716)
('green', 2018, 8765558)
('green', 2019, 6120937)
('green', 2020, 1663021)
('green', 2021, 1023443)
('green', 2022, 786741)
('green', 2023, 743961)
('green', 2024, 622572)
('green', 2025, 561485)
('yellow', 2015, 145021913)
('yellow', 2016, 130211223)
('yellow', 2017, 112622241)
('yellow', 2018, 102030706)
('yellow', 2019, 83634131)
('yellow', 2020, 24191724)
('yellow', 2021, 30289990)
('yellow', 2022, 38805145)
('yellow', 2023, 37175135)

--- sample_metrics ---
('yellow', 0.7, 6.8, 3.63, 11.56, 20.0)
('yellow', 0.8, 5.8, 3.25, 14.77, 0.0)
('yellow', 5.1, 16.3, 9.37, 32.67, 0.0)
('yellow', 3.2, 14.05, 10.57, 18.17, 15.22)
('yellow', 1.2, 7.3, 6.12, 11.77, 0.0)
('yellow', 0.8, 6.95, 3.53, 13.58, 23.0)
('yellow', 3.0, 10.8, 7.15, 25.17, 0.0)
('yellow', 3.56, 14.8, 11.55, 18.49, 7.69)
('yello

Cerramos el cursor y el conector a Snowflake con seguridad.

In [14]:
cur.close()
conn.close()
print("Notebook 03 finished")

Notebook 03 finished
